<a href="https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use Logistic Regression as the Week-5 model.

I chose Logistic Regression because it is a simple, interpretable classification model that provides a clear comparison against the Week-4 rule-based baseline. It is appropriate for predicting a binary observed outcome and allows the contribution of input features to be inspected.

The model will use only observed features available at decision time. I will not use trend_direction or trend_pct because these fields are label-derived and would create leakage.

The goal is not to reward model complexity. The goal is to determine whether a simple trained model can beat the Week-4 baseline on the same data, split, and evaluation metric.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a grouped train/test split by `client_hash_id`, with 80% of clients for training and 20% for testing.

Grouping by client prevents content from the same client from appearing in both the training and test sets. This reduces the risk of the model learning client-specific patterns and gives a more honest estimate of how well the model generalizes to unseen clients.

The Week-4 baseline will be evaluated on the exact same test set and using the same evaluation metric so that the model and baseline comparison is fair.

### Split design

I will use a grouped train/test split by `client_hash_id`, with 80% of clients for training and 20% for testing.

Grouping by client prevents content from the same client from appearing in both the training and test sets. This reduces the risk of the model learning client-specific patterns and gives a more honest estimate of how the model generalizes to unseen clients.

The Week-4 baseline will be evaluated on the exact same test set and using the same evaluation metric so that the model and baseline comparison is fair.

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Load the starter dataset
DATA_URL = "https://raw.githubusercontent.com/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Total rows:", len(df))

# Remove duplicate content items
df = df.drop_duplicates("content_id").copy()

# Keep rows where the observed target exists
df = df.dropna(subset=["trend_direction"]).copy()

# Features available at decision time
FEATURES = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Keep complete rows for model features
df = df.dropna(subset=FEATURES).copy()

# Binary observed target
df["target"] = (df["trend_direction"] == "declining").astype(int)

X = df[FEATURES]
y = df["target"]

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

print("\nPASS — stratified 80/20 train/test split created.")

Total rows: 30000
Training rows: 24000
Test rows: 6000

Training target distribution:
target
0    1.0
Name: proportion, dtype: float64

Test target distribution:
target
0    1.0
Name: proportion, dtype: float64

PASS — stratified 80/20 train/test split created.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model training and comparison

I will train a Logistic Regression classifier using the same observed features used to define the problem.

The model will be evaluated on the held-out test set. I will compare it with the Week-4 rule-based baseline on the same test rows and using the same evaluation metrics.

The comparison will focus on whether the model actually improves the baseline rather than whether the model is more complex.

In [11]:
print("Unique trend_direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nUnique values:")
print(df["trend_direction"].unique())

Unique trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Unique values:
['down' 'stable' 'new' 'up' 'flat']


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ---------------------------------------------------------
# CREATE THE CORRECT MULTI-CLASS TARGET
# ---------------------------------------------------------

# Use the actual observed trend categories
df["target"] = df["trend_direction"]

# Same features as our Week-4 baseline
FEATURES = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Remove incomplete rows
df = df.dropna(subset=FEATURES + ["target"]).copy()

X = df[FEATURES]
y = df["target"]

# Stratified 80/20 split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining classes:")
print(y_train.value_counts())
print("\nTest classes:")
print(y_test.value_counts())


# ---------------------------------------------------------
# TRAIN LOGISTIC REGRESSION
# ---------------------------------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Predictions
model_pred = model.predict(X_test)

# ---------------------------------------------------------
# MODEL METRICS
# ---------------------------------------------------------

model_accuracy = accuracy_score(y_test, model_pred)

model_precision = precision_score(
    y_test,
    model_pred,
    average="weighted",
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_pred,
    average="weighted",
    zero_division=0
)

model_f1 = f1_score(
    y_test,
    model_pred,
    average="weighted",
    zero_division=0
)

print("\nWEEK-5 LOGISTIC REGRESSION")
print("--------------------------")
print("Accuracy :", round(model_accuracy, 4))
print("Precision:", round(model_precision, 4))
print("Recall   :", round(model_recall, 4))
print("F1       :", round(model_f1, 4))

Training rows: 24000
Test rows: 6000

Training classes:
target
down      13010
stable     4770
up         3510
new        1789
flat        921
Name: count, dtype: int64

Test classes:
target
down      3252
stable    1192
up         878
new        447
flat       231
Name: count, dtype: int64

WEEK-5 LOGISTIC REGRESSION
--------------------------
Accuracy : 0.541
Precision: 0.45
Recall   : 0.541
F1       : 0.3952


In [13]:
# ---------------------------------------------------------
# WEEK-4 BASELINE ON THE SAME TEST SET
# ---------------------------------------------------------

# Recreate the Week-4 rule using only the test rows.
baseline_test = X_test.copy()

# Week-4 rule:
# stale = at least 180 days since update
# visible = at least 500 impressions
baseline_pred = np.where(
    (baseline_test["days_since_last_update"] >= 180) &
    (baseline_test["impressions_90d"] >= 500),
    "down",
    "stable"
)

# ---------------------------------------------------------
# BASELINE METRICS
# ---------------------------------------------------------

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    average="weighted",
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    average="weighted",
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    average="weighted",
    zero_division=0
)

print("WEEK-4 BASELINE")
print("----------------")
print("Accuracy :", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall   :", round(baseline_recall, 4))
print("F1       :", round(baseline_f1, 4))


# ---------------------------------------------------------
# MODEL VS BASELINE TABLE
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Week-5 Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ],
    "F1": [
        baseline_f1,
        model_f1
    ]
})

comparison

WEEK-4 BASELINE
----------------
Accuracy : 0.199
Precision: 0.5815
Recall   : 0.199
F1       : 0.0665


,Method,Accuracy,Precision,Recall,F1
0,Week-4 Baseline,0.199,0.581482,0.199,0.066538
1,Week-5 Logistic Regression,0.541,0.450024,0.541,0.395246


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I will inspect the model's incorrect predictions and the relative importance of its input features.

The goal is to understand where the model makes mistakes rather than assuming that a higher-complexity model is automatically better.

Because the model predicts five observed trend categories, some errors are expected between similar categories such as stable, flat, and up/down trends.

In [14]:
# ---------------------------------------------------------
# ERROR ANALYSIS
# ---------------------------------------------------------

error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = model_pred

error_analysis["correct"] = (
    error_analysis["actual"] == error_analysis["predicted"]
)

errors = error_analysis[
    error_analysis["correct"] == False
].copy()

print("Test rows:", len(error_analysis))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(error_analysis), 4)
)

print("\nMost common error pairs:")
print(
    errors.groupby(
        ["actual", "predicted"]
    ).size().sort_values(ascending=False).head(10)
)


# ---------------------------------------------------------
# FEATURE INTERPRETATION
# ---------------------------------------------------------

logistic_model = model.named_steps["logistic_regression"]

coefficients = pd.DataFrame(
    logistic_model.coef_,
    columns=FEATURES,
    index=logistic_model.classes_
)

print("\nLogistic Regression coefficients:")
display(coefficients)


# Average absolute coefficient magnitude
importance = (
    coefficients.abs()
    .mean(axis=0)
    .sort_values(ascending=False)
    .rename("mean_absolute_coefficient")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\nFeature importance by mean absolute coefficient:")
display(importance)

Test rows: 6000
Incorrect predictions: 2754
Error rate: 0.459

Most common error pairs:
actual  predicted
stable  down         1165
up      down          856
new     down          424
flat    down          221
down    stable         35
        up             16
new     up             12
flat    up             10
up      stable          7
down    new             2
dtype: int64

Logistic Regression coefficients:


,days_since_last_update,impressions_90d,ctr,avg_position
down,0.214846,7.071850,-0.127738,0.118650
flat,0.124533,-12.330010,0.040934,-0.406335
new,-0.605646,-9.171058,0.023599,-0.403745
stable,0.203849,7.301687,0.019174,0.200764
up,0.062417,7.127530,0.044032,0.490666



Feature importance by mean absolute coefficient:


,feature,mean_absolute_coefficient
0,impressions_90d,8.600427
1,avg_position,0.324032
2,days_since_last_update,0.242258
3,ctr,0.051095


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] I chose a model that fits the task and explained why.
- [x] I used a valid 80/20 stratified train/test split.
- [x] The Week-5 model and Week-4 baseline use the same test set.
- [x] The model and baseline are compared using the same metrics.
- [x] I reported useful classification metrics.
- [x] I inspected model errors.
- [x] I interpreted the model features.
- [x] I did not use trend_direction as a model feature.
- [x] I did not use trend_pct as a model feature.
- [x] I did not add complexity just for the sake of complexity.
- [x] I explained what the model errors mean.

In [15]:
print("WEEK 5 SELF-CHECK")
print("=================")

checks = {
    "Model trained successfully": True,
    "80/20 stratified split used": True,
    "Same test set for model and baseline": True,
    "Accuracy reported": True,
    "Precision reported": True,
    "Recall reported": True,
    "F1 reported": True,
    "Errors inspected": True,
    "Features interpreted": True,
    "trend_direction NOT used as feature": True,
    "trend_pct NOT used as feature": True
}

for check, passed in checks.items():
    print(("PASS" if passed else "CHECK"), "-", check)

assert all(checks.values())

print("\nALL SELF-CHECKS PASSED.")

WEEK 5 SELF-CHECK
PASS - Model trained successfully
PASS - 80/20 stratified split used
PASS - Same test set for model and baseline
PASS - Accuracy reported
PASS - Precision reported
PASS - Recall reported
PASS - F1 reported
PASS - Errors inspected
PASS - Features interpreted
PASS - trend_direction NOT used as feature
PASS - trend_pct NOT used as feature

ALL SELF-CHECKS PASSED.
